<a href="https://colab.research.google.com/github/35Rony/AI_SE/blob/main/Task3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Import necessary libraries
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report, accuracy_score, f1_score
import numpy as np
import os
import pandas as pd

# Parameters
img_size = (224, 224)
batch_size = 32
# Assuming a binary classification problem based on the available files (mask/no mask)
num_classes = 2  # Adjust based on your actual classes

# Data preparation: Create a DataFrame with image paths and labels
# This assumes a binary classification where images with 'mask' in the filename
# belong to one class and others to another. Adjust this logic if your
# classification criteria are different.
image_dir = '/content/'
filepaths = [os.path.join(image_dir, f) for f in os.listdir(image_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
# Basic labeling based on filename content - adjust as needed
labels = ['mask' if 'mask' in os.path.basename(f) else 'no_mask' for f in filepaths]

image_df = pd.DataFrame({'filepaths': filepaths, 'labels': labels})

# Use ImageDataGenerator for data augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    validation_split=0.2  # 20% validation split
)

# Use flow_from_dataframe as files are not in class subdirectories
train_generator = train_datagen.flow_from_dataframe(
    dataframe=image_df,
    x_col='filepaths',
    y_col='labels',
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='training'
)

val_generator = train_datagen.flow_from_dataframe(
    dataframe=image_df,
    x_col='filepaths',
    y_col='labels',
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation'
)

# Load pretrained ResNet50 without the final classification layer
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=img_size + (3,))
x = Flatten()(base_model.output)
output = Dense(num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)

# Freeze base model layers to transfer learn only the final layers initially
for layer in base_model.layers:
    layer.trainable = False

# Compile the model
model.compile(optimizer=Adam(learning_rate=1e-4), loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10
)

# Evaluate model on validation set and calculate F1-score
val_generator.reset()
# Get true labels directly from the generator's dataframe subset in the correct order
y_true = val_generator.classes[val_generator.index_array]

# Predict on the validation data
y_pred_prob = model.predict(val_generator)
y_pred = np.argmax(y_pred_prob, axis=1)

accuracy = accuracy_score(y_true, y_pred)
# Get target names from the generator's class_indices
target_names = list(val_generator.class_indices.keys())
f1 = f1_score(y_true, y_pred, average='weighted')

print(f'Validation Accuracy: {accuracy:.4f}')
print(f'Validation F1-score: {f1:.4f}')

# Optionally, generate full classification report
print(classification_report(y_true, y_pred, target_names=target_names))

Found 160 validated image filenames belonging to 2 classes.
Found 40 validated image filenames belonging to 2 classes.
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 49s 9s/step - accuracy: 0.5504 - loss: 0.7987 - val_accuracy: 0.8500 - val_loss: 0.4594
Epoch 2/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 38s 8s/step - accuracy: 0.8282 - loss: 0.4567 - val_accuracy: 0.9500 - val_loss: 0.3433
Epoch 3/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 38s 8s/step - accuracy: 0.9655 - loss: 0.3045 - val_accuracy: 0.9500 - val_loss: 0.2231
Epoch 4/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 38s 8s/step - accuracy: 0.9406 - loss: 0.2206 - val_accuracy: 0.9250 - val_loss: 0.1940
Epoch 5/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 40s 9s/step - accuracy: 0.9382 - loss: 0.1996 - val_accuracy: 1.0000 - val_loss: 0.1689
Epoch 6/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 38s 8s/step - accuracy: 0.9642 - loss: 0.1622 - val_accuracy: 0.9500 - val_loss: 0.1489
Epoch 7/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 41s 8s/step - accuracy: 0.9486 - loss: 0.1696 - val_accuracy: 0.9750 - val_loss: 0.1293
Epoch 8/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 38s 8s/step - accuracy: 0.9702 - loss: 0.1193 - val_accuracy: 0.9750 - val_loss: 0.1410
Epoch 9/